# Part 0: Setup

In [ ]:
# Libraries installation

# [ --- fill code --- ]
!pip install -q langchain-community langchain-text-splitters sentence-transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.documents import Document

import numpy as np
from typing import List
import time
import re

## 0.1 Download Workshop File



In [ ]:
# Download pdf file
!gdown 1m1OI12Hrcs4kOJ8i56_iq4xGQ84QFoFz -O Newton-biography.pdf

Downloading...
From: https://drive.google.com/uc?id=1m1OI12Hrcs4kOJ8i56_iq4xGQ84QFoFz
To: /content/Newton-biography.pdf
100% 26.9k/26.9k [00:00<00:00, 71.8MB/s]


# Part 1: Text Splitters

##1.1 Load Documents

In [ ]:
# Initialize loader for the specific PDF
pdf_loader = PyPDFLoader("Newton-biography.pdf")

# Load document content into a list of pages
newton_docs = pdf_loader.load()

# Verify output: Page count, total characters, and content preview
print(f"Loaded {len(newton_docs)} pages")
print(f"Total characters: {sum(len(doc.page_content) for doc in newton_docs):,}")
print(f"Preview: {newton_docs[0].page_content[:200].replace(chr(10), ' ')}...")

Loaded 6 pages
Total characters: 22,463
Preview: Sir Isaac Newton    Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England   Died: 31 March 1727 in London, England    Isaac Newton's life can be divided into three quite distinct periods. The first i...


In [ ]:
newton_docs[0].page_content[:200]

"Sir Isaac Newton \n \nBorn: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England  \nDied: 31 March 1727 in London, England \n \nIsaac Newton's life can be divided into three quite distinct periods. The first i"

In [ ]:
newton_docs

[Document(metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Sir Isaac Newton \n \nBorn: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England  \nDied: 31 March 1727 in London, England \n \nIsaac Newton\'s life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up \nto his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in \nwhich he was Lucasian professor at Cambridge. The third period (nearly as long as the other two combined) \nsaw Newton as a highly paid government official in London with little further interest in mathematical research.  \nIsaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the

## 1.2 Data Exploration

In [ ]:
# --- Data Exploration ---

# 1. Check data type (Expected: list)
print(f"Data type: {type(newton_docs)}")

# 2. Get total page count
print(f"Total pages: {len(newton_docs)}")

print("\n--- Exploring the First Page ---")
# 3. Access the first page (Index 0)
first_page = newton_docs[0]

# 4. Inspect object structure (Metadata + Page Content)
print(f"Object structure: {first_page}")

# 5. Extract Metadata for citation and source tracking
print(f"\nMetadata of page 1: {first_page.metadata}")
print(f"Source file: {first_page.metadata['source']}")
print(f"Page number: {first_page.metadata['page']}")

# 6. Analyze Page Content and character count
content_length = len(first_page.page_content)
print(f"\nContent length on page 1: {content_length} characters")
print(f"First 100 characters: {first_page.page_content[:100]}...")

Data type: <class 'list'>
Total pages: 6

--- Exploring the First Page ---
Object structure: page_content='Sir Isaac Newton 
 
Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England  
Died: 31 March 1727 in London, England 
 
Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up 
to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in 
which he was Lucasian professor at Cambridge. The third period (nearly as long as the other two combined) 
saw Newton as a highly paid government official in London with little further interest in mathematical research.  
Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the 
calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 
in this biography which is the "corrected" Gregorian calendar date bringing it into line with our present 
ca

In [ ]:
newton_docs[1]

Document(metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}, page_content='We know nothing about what Isaac learnt in preparation for university, but Stokes was an able man and almost \ncertainly gave Isaac private coaching and a good grounding. There is no evidence that he learnt any \nmathematics, but we cannot rule out Stokes introducing him to Euclid\'s Elements which he was well capable of \nteaching (although there is evidence mentioned below that Newton did not read Euclid before 1663). Anecdotes \nabound about a mechanical ability which Isaac displayed at the school and stories are told of his skill in making \nmodels of machines, in particular of clocks and windmills. However, when biographers seek information about \nfamous people there

## 1.3 Validate PDF quality

In [ ]:
def inspect_document(docs: list) -> None:

    # Merge all pages into a single string for global analysis
    full_text = " ".join([d.page_content for d in docs])
    total_chars = len(full_text)

    print("=== Document Health Check ===")
    print(f"Pages           : {len(docs)}")
    print(f"Total chars     : {total_chars}")

    # Handle edge case: empty document
    if total_chars == 0:
        print("\n⚠️ Empty document — nothing to inspect.")
        return

    # Count formatting issues: Null bytes, triple spaces, and triple newlines
    null_bytes    = full_text.count('\x00')
    excess_spaces = len(re.findall(r' {3,}', full_text))
    excess_lines  = len(re.findall(r'\n{3,}', full_text))

    # Display global summary with status indicators
    print(f"\nNull bytes      : {null_bytes}   {'⚠️ Action: Clean needed' if null_bytes > 0 else '✅ None found'}")
    print(f"Excess spaces   : {excess_spaces}   {'⚠️ Action: Clean needed' if excess_spaces > 0 else '✅ None found'}")
    print(f"Excess newlines : {excess_lines}   {'⚠️ Action: Clean needed' if excess_lines > 0 else '✅ None found'}")

    # Analyze each page individually to pinpoint issues
    print("\n--- Per-page Breakdown ---")
    for i, doc in enumerate(docs):
        p = doc.page_content
        flags = []
        # Check specific flags per page
        if p.count('\x00') > 0: flags.append("null bytes")
        if len(re.findall(r' {3,}', p)) > 0: flags.append("excess spaces")
        if len(re.findall(r'\n{3,}', p))> 0: flags.append("excess newlines")

        status = ("⚠️  [" + ", ".join(flags) + "]") if flags else "✅ Clean"
        print(f"  Page {i+1}: {status}")

    # Aggregate all detected issues for the final verdict
    issues = []
    if null_bytes    > 0: issues.append("Null bytes")
    if excess_spaces > 0: issues.append("Excess spaces")
    if excess_lines  > 0: issues.append("Excess newlines")

    print()
    if issues:
        print(f"Conclusion: ⚠️ Cleaning recommended — Issues: {', '.join(issues)}")
    else:
        print("Conclusion: ✅ Document looks clean")

In [ ]:
inspect_document(newton_docs)

=== Document Health Check ===
Pages           : 6
Total chars     : 22468

Null bytes      : 0   ✅ None found
Excess spaces   : 0   ✅ None found
Excess newlines : 0   ✅ None found

--- Per-page Breakdown ---
  Page 1: ✅ Clean
  Page 2: ✅ Clean
  Page 3: ✅ Clean
  Page 4: ✅ Clean
  Page 5: ✅ Clean
  Page 6: ✅ Clean

Conclusion: ✅ Document looks clean


In [ ]:
from langchain_core.documents import Document

newton_docs_sample = [
    Document(
        page_content=(
            # 1. Null bytes: Caused by corrupted export or binary artifacts
            "Isaac Newton\x00 was born on 25 December 1642\x00 in Woolsthorpe, Lincolnshire.\n"
            "He is widely recognised as one of the greatest mathematicians\x00 and physicists.\n"
        ),
        metadata={"source": "newton_bio.pdf", "page": 1}
    ),
    Document(
        page_content=(
            # 2. Excess spaces: Results from column layouts or tab-based formatting
            "Newton developed   the   laws   of   motion   and   universal   gravitation.\n"
            "His   work   Philosophiæ   Naturalis   Principia   Mathematica   (1687)\n"
            "laid   the   foundations   for   classical   mechanics        and   physics.\n"
        ),
        metadata={"source": "newton_bio.pdf", "page": 2}
    ),
    Document(
        page_content=(
            # 3. Excess newlines: Caused by hard section breaks or headers/footers
            "Beyond mechanics, Newton made contributions to optics.\n\n\n\n"
            "He discovered that white light is composed of a spectrum of colours.\n\n\n\n\n"
            "Newton also developed calculus independently of Leibniz.\n\n\n\n"
            "He served as Warden and later Master of the Royal Mint.\n"
        ),
        metadata={"source": "newton_bio.pdf", "page": 3}
    ),
]

# Run the health check on the synthetic data
inspect_document(newton_docs_sample)

=== Document Health Check ===
Pages           : 3
Total chars     : 628

Null bytes      : 3   ⚠️ Action: Clean needed
Excess spaces   : 20   ⚠️ Action: Clean needed
Excess newlines : 3   ⚠️ Action: Clean needed

--- Per-page Breakdown ---
  Page 1: ⚠️  [null bytes]
  Page 2: ⚠️  [excess spaces]
  Page 3: ⚠️  [excess newlines]

Conclusion: ⚠️ Cleaning recommended — Issues: Null bytes, Excess spaces, Excess newlines


In [ ]:
def clean_document(docs: list) -> list:

    cleaned_docs = []

    for doc in docs:
        text = doc.page_content

        # 1. Remove Null bytes (binary artifacts often found in PDFs/scans)
        text = text.replace('\x00', '')

        # 2. Collapse large clusters of spaces (3+) into a single space
        text = re.sub(r' {3,}', ' ', text)

        # 3. Normalize vertical whitespace: Reduce excessive newlines (3+) to double newlines
        text = re.sub(r'\n{3,}', '\n\n', text)

        # 4. Convert single line-wraps (\n) into spaces
        text = text.replace('\n\n', '<<PARA>>')
        text = text.replace('\n', ' ')
        text = text.replace('<<PARA>>', '\n\n')

        # 5. Clean up any double spaces that may have been created during conversion
        text = re.sub(r' {2,}', ' ', text)

        # Reconstruct Document with sanitized content and original metadata
        cleaned_docs.append(
            Document(
                page_content=text,
                metadata=doc.metadata
            )
        )

    return cleaned_docs

In [ ]:
# Clean sample document
cleaned_newton_docs_sample = clean_document(newton_docs_sample)

In [ ]:
cleaned_newton_docs_sample

[Document(metadata={'source': 'newton_bio.pdf', 'page': 1}, page_content='Isaac Newton was born on 25 December 1642 in Woolsthorpe, Lincolnshire. He is widely recognised as one of the greatest mathematicians and physicists. '),
 Document(metadata={'source': 'newton_bio.pdf', 'page': 2}, page_content='Newton developed the laws of motion and universal gravitation. His work Philosophiæ Naturalis Principia Mathematica (1687) laid the foundations for classical mechanics and physics. '),
 Document(metadata={'source': 'newton_bio.pdf', 'page': 3}, page_content='Beyond mechanics, Newton made contributions to optics.\n\nHe discovered that white light is composed of a spectrum of colours.\n\nNewton also developed calculus independently of Leibniz.\n\nHe served as Warden and later Master of the Royal Mint. ')]

In [ ]:
newton_docs_sample

[Document(metadata={'source': 'newton_bio.pdf', 'page': 1}, page_content='Isaac Newton\x00 was born on 25 December 1642\x00 in Woolsthorpe, Lincolnshire.\nHe is widely recognised as one of the greatest mathematicians\x00 and physicists.\n'),
 Document(metadata={'source': 'newton_bio.pdf', 'page': 2}, page_content='Newton developed   the   laws   of   motion   and   universal   gravitation.\nHis   work   Philosophiæ   Naturalis   Principia   Mathematica   (1687)\nlaid   the   foundations   for   classical   mechanics        and   physics.\n'),
 Document(metadata={'source': 'newton_bio.pdf', 'page': 3}, page_content='Beyond mechanics, Newton made contributions to optics.\n\n\n\nHe discovered that white light is composed of a spectrum of colours.\n\n\n\n\nNewton also developed calculus independently of Leibniz.\n\n\n\nHe served as Warden and later Master of the Royal Mint.\n')]

In [ ]:
inspect_document(newton_docs_sample)

=== Document Health Check ===
Pages           : 3
Total chars     : 628

Null bytes      : 3   ⚠️ Action: Clean needed
Excess spaces   : 20   ⚠️ Action: Clean needed
Excess newlines : 3   ⚠️ Action: Clean needed

--- Per-page Breakdown ---
  Page 1: ⚠️  [null bytes]
  Page 2: ⚠️  [excess spaces]
  Page 3: ⚠️  [excess newlines]

Conclusion: ⚠️ Cleaning recommended — Issues: Null bytes, Excess spaces, Excess newlines


In [ ]:
# inspect sample document
inspect_document(cleaned_newton_docs_sample)

=== Document Health Check ===
Pages           : 3
Total chars     : 573

Null bytes      : 0   ✅ None found
Excess spaces   : 0   ✅ None found
Excess newlines : 0   ✅ None found

--- Per-page Breakdown ---
  Page 1: ✅ Clean
  Page 2: ✅ Clean
  Page 3: ✅ Clean

Conclusion: ✅ Document looks clean


In [ ]:
newton_docs[0].page_content
# Problem: A new line appears but the sentence is incomplete.

'Sir Isaac Newton \n \nBorn: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England  \nDied: 31 March 1727 in London, England \n \nIsaac Newton\'s life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up \nto his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in \nwhich he was Lucasian professor at Cambridge. The third period (nearly as long as the other two combined) \nsaw Newton as a highly paid government official in London with little further interest in mathematical research.  \nIsaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the \ncalendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 \nin this biography which is the "corrected" Gregorian calendar date bringing it into line with our present \ncalendar. (The Gregorian calendar was not adopted in England until 1752.) Isaac Newton came fro

In [ ]:
# clean Newton document
cleaned_newton_docs = clean_document(newton_docs)

In [ ]:
cleaned_newton_docs[0].page_content

'Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton\'s life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge. The third period (nearly as long as the other two combined) saw Newton as a highly paid government official in London with little further interest in mathematical research. Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography which is the "corrected" Gregorian calendar date bringing it into line with our present calendar. (The Gregorian calendar was not adopted in England until 1752.) Isaac Newton came from a family of farmers but ne

## 1.4 Chunk

### 1.4.1 Character Text Splitter




In [ ]:
# Standard Splitting: No Overlap
no_overlap_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=200,
    chunk_overlap=0,
    length_function=len,
)

# Contextual Splitting: With Overlap
with_overlap_splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=200,
    chunk_overlap=40,
    length_function=len,
)

# Execute splitting on the source documents
no_overlap_chunks = no_overlap_splitter.split_documents(cleaned_newton_docs)
with_overlap_chunks = with_overlap_splitter.split_documents(cleaned_newton_docs)

# Output comparison results
print(f"No overlap    : {len(no_overlap_chunks)} chunks")
print(f"With overlap  : {len(with_overlap_chunks)} chunks")

# Calculate delta to show how overlap increases chunk count for better retrieval
print(f"Difference    : +{len(with_overlap_chunks) - len(no_overlap_chunks)} extra chunks for context preservation")

No overlap    : 115 chunks
With overlap  : 141 chunks
Difference    : +26 extra chunks for context preservation


no_overlap_chunks

In [ ]:
no_overlap_chunks[0].page_content

"Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his"

In [ ]:
no_overlap_chunks[1].page_content

'boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge. The third period'

with_overlap_chunks

In [ ]:
with_overlap_chunks[0].page_content

"Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his"

In [ ]:
with_overlap_chunks[1].page_content

'quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian'

### 1.4.2 Recursive Character Text Splitter

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=150,   # 40-character buffer for context continuity
    separators=["\n\n", "\n", ". ", ""], # Priority list for splitting points
    keep_separator=True
)

# Process the first page to observe the chunking behavior
recursive_chunks = recursive_splitter.split_documents(cleaned_newton_docs)


In [ ]:
recursive_chunks[0].page_content

"Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669"

In [ ]:
recursive_chunks[1].page_content

'. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge'

In [ ]:
mini_recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=85,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", ""],
    keep_separator=True
)

# Process the first page to observe the chunking behavior
mini_recursive_chunks = mini_recursive_splitter.split_documents(cleaned_newton_docs)

In [ ]:
mini_recursive_chunks[0].page_content

'Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 Marc'

In [ ]:
mini_recursive_chunks[1].page_content

"rpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life"

# PART 2: EMBEDDINGS

## 2.1 Initialize embedding model

In [ ]:
# Load the model to convert text into numerical vectors (embeddings)

start_time = time.time()

embeddings = HuggingFaceEmbeddings(
    # Use a popular 'free' and 'fast' model (384 dimensions or 256 Tokens)
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    # model_name="sentence-transformers/all-mpnet-base-v2",

    # Force execution on CPU to ensure compatibility across all environments without a GPU
    model_kwargs={"device": "cpu"},

    # Normalize embeddings to a standard scale to improve similarity calculation accuracy
    encode_kwargs={"normalize_embeddings": True},
)

load_time = time.time() - start_time


# Test the embedding process by converting a sample sentence into a vector
test_vector = embeddings.embed_query("Newton discovered gravity")

# Calculate the Vector Magnitude
magnitude = sum(v**2 for v in test_vector) ** 0.5

print(f"Model status      : Successfully loaded!")
print(f"Loading time      : {load_time:.1f} seconds")
print(f"Vector dimensions : {len(test_vector)} numbers per sentence")
print(f"Check Magnitude   : {magnitude:.4f} (If it's 1.0, it's perfect)")

/tmp/ipykernel_10728/3377825173.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model status      : Successfully loaded!
Loading time      : 14.5 seconds
Vector dimensions : 384 numbers per sentence
Check Magnitude   : 1.0000 (If it's 1.0, it's perfect)


## 2.2 Embed Newton Chunks

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

recursive_chunks = recursive_splitter.split_documents(cleaned_newton_docs)


In [ ]:
# Embed Newton Chunks
# Convert Newton's text chunks into numerical representations (Vectors)

# Extract raw text from each Chunk object
newton_texts = [chunk.page_content for chunk in recursive_chunks]

start_time = time.time()

# embeddings raw text list
newton_embeddings = embeddings.embed_documents(newton_texts)

embed_time = time.time() - start_time

# Display performance and execution metrics
print(f"\n=== Newton Embeddings Summary ===")
print(f"Chunks embedded : {len(newton_embeddings)} items")
print(f"Dimensions      : {len(newton_embeddings[0])} (Length of each vector)")
print(f"Time taken      : {embed_time:.2f} seconds")
print(f"Processing Speed: {len(newton_embeddings) / embed_time:.1f} chunks/second")

# Perform a statistical sanity check on the first vector
first_vec = np.array(newton_embeddings[12])
print(f"\nVector Statistics (First chunk):")
print(f"  Mean : {first_vec.mean():.4f} (Average value)")
print(f"  Std  : {first_vec.std():.4f}  (Numerical spread/Standard Deviation)")
print(f"  Min  : {first_vec.min():.4f}")
print(f"  Max  : {first_vec.max():.4f}")


=== Newton Embeddings Summary ===
Chunks embedded : 161 items
Dimensions      : 384 (Length of each vector)
Time taken      : 2.14 seconds
Processing Speed: 75.2 chunks/second

Vector Statistics (First chunk):
  Mean : -0.0006 (Average value)
  Std  : 0.0510  (Numerical spread/Standard Deviation)
  Min  : -0.1389
  Max  : 0.1363


In [ ]:
first_vec = np.array(newton_embeddings[10])
print(f"\nVector Statistics (First chunk):")
print(f"  Mean : {first_vec.mean():.4f} (Average value)")
print(f"  Std  : {first_vec.std():.4f}  (Numerical spread/Standard Deviation)")
print(f"  Min  : {first_vec.min():.4f}")
print(f"  Max  : {first_vec.max():.4f}")


Vector Statistics (First chunk):
  Mean : -0.0006 (Average value)
  Std  : 0.0510  (Numerical spread/Standard Deviation)
  Min  : -0.1712
  Max  : 0.1755


In [ ]:
# Check the structure of the embedding matrix
np.array(newton_embeddings).shape

(161, 384)

In [ ]:
# the number of chunk
len(newton_texts)

161

In [ ]:
newton_texts

["Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods",
 '. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge',
 '. The third period (nearly as long as the other two combined) saw Newton as a highly paid government official in London with little further interest in mathematical research',
 '. Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire',
 '. Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography which is the "corrected" Gregorian calendar date',
 'the "corrected" Gregorian calendar date bringing it into line with our present calendar',
 '. (The Gregorian calendar was not adopted in E

In [ ]:
newton_embeddings

[[0.005715386010706425,
  -0.08080600202083588,
  -0.01825558952987194,
  -0.0009799086255952716,
  0.034510672092437744,
  0.07749779522418976,
  -0.07019230723381042,
  0.005772048141807318,
  -0.052754636853933334,
  0.06162285432219505,
  -0.031184690073132515,
  -0.04928496852517128,
  -0.044316988438367844,
  -0.03737657517194748,
  -0.02747330255806446,
  0.05834190174937248,
  -0.14783905446529388,
  -0.02770642563700676,
  -0.04600295051932335,
  0.025393707677721977,
  -0.015318702906370163,
  0.03578624874353409,
  -0.013274000026285648,
  -0.0050626397132873535,
  0.03790128603577614,
  0.07963850349187851,
  -0.036911606788635254,
  -0.050834402441978455,
  0.04144938662648201,
  0.07481759786605835,
  0.1097191795706749,
  -0.052140507847070694,
  0.09488943219184875,
  -0.030184002593159676,
  -0.09724315255880356,
  0.046784624457359314,
  0.008105969056487083,
  -0.03798312321305275,
  0.046858880668878555,
  -0.01235683262348175,
  -0.07426965236663818,
  -0.001373554

In [ ]:
# Calculate the Vector Magnitude
magnitude = sum(v**2 for v in newton_embeddings[0]) ** 0.5
magnitude

1.0000000452712765

#Part 3: Chunking Strategy
การทดลองเปรียบเทียบกลยุทธ์การหั่นข้อมูล (Chunking Strategy Lab)

## 3.1 Define strategies

In [ ]:
strategies = {
    # --- SMALL CHUNKS ---
    "Small / No overlap"  : RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""], chunk_size=100, chunk_overlap=0),

    "Small / With overlap": RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""], chunk_size=100, chunk_overlap=50),

    # --- MEDIUM CHUNKS ---
    "Medium / No overlap" : RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""], chunk_size=300, chunk_overlap=0),

    "Medium / With overlap": RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""], chunk_size=300, chunk_overlap=150),

    # --- LARGE CHUNKS ---
    "Large / No overlap"  : RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""], chunk_size=500, chunk_overlap=0),

    "Large / With overlap": RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""], chunk_size=500, chunk_overlap=250),
}

## 3.2 Splitting and Embedding

In [ ]:
results = {}

print("=== Slicing & Vectorizing Progress ===\n")

print(f"{'Strategy':<25} {'Chunks':>6} {'Avg chars':>10} {'Time':>8}")
print("-" * 55)

for name, splitter in strategies.items():
    # Perform Document Splitting (Chunking)
    chunks = splitter.split_documents(cleaned_newton_docs)
    texts = [c.page_content for c in chunks]

    # Convert text to Vector Representations (Embedding) and track duration
    start = time.time()
    chunk_embs = embeddings.embed_documents(texts)
    elapsed = time.time() - start

    # Calculate Average Chunk Length (to verify splitting quality)
    avg_len = sum(len(t) for t in texts) / len(texts)

    # Store results in a dictionary for the subsequent Evaluation step
    results[name] = {
        "chunks"    : chunks,
        "embeddings": chunk_embs,
        "avg_len"   : avg_len,
    }

    # Log the performance and output metrics for each strategy
    print(f"{name:<25} {len(chunks):>6} {avg_len:>10.0f} {elapsed:>6.2f}s")

=== Slicing & Vectorizing Progress ===

Strategy                  Chunks  Avg chars     Time
-------------------------------------------------------
Small / No overlap           304         72   2.25s
Small / With overlap         363         86   2.35s
Medium / No overlap          101        220   1.49s
Medium / With overlap        119        233   1.82s
Large / No overlap            55        403   1.59s
Large / With overlap          83        421   2.32s


In [ ]:
s_no = results["Small / No overlap"]["chunks"]
s_with = results["Small / With overlap"]["chunks"]

# s_with = results["Small / With overlap"]["chunks"][45].page_content
# print("=== Content Comparison ===")

print("Small / No overlap : ")
print(s_no[44].page_content)
print("-"*50)
print(s_no[45].page_content)
print("\nSmall / With overlap : ")
print(s_with[45].page_content)
print("-"*50)
print(s_with[46].page_content)

Small / No overlap : 
is likely that Isaac had shown more promise in his first spell at the school than the school
--------------------------------------------------
reports suggest

Small / With overlap : 
his mother that this was the right thing to do, Isaac was allowed to return to the Free Grammar
--------------------------------------------------
Isaac was allowed to return to the Free Grammar School in Grantham in 1660 to complete his school


In [ ]:
m_no = results["Medium / No overlap"]["chunks"][1].page_content
m_with = results["Medium / With overlap"]["chunks"][1].page_content

# print("=== Content Comparison ===")
print("Medium no overlap : ")
print(m_no)
print("\nMedium with overlap : ")
print(m_with)

Medium no overlap : 
. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge. The third period (nearly as long as the other two combined) saw Newton as a highly paid government official in London with little further interest in mathematical research

Medium with overlap : 
. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge


In [ ]:
m_no = results["Medium / No overlap"]["chunks"]
m_with = results["Medium / With overlap"]["chunks"]

# print("=== Content Comparison ===")
print("Medium no overlap : ")
print(m_no[0].page_content)
print("-"*50)
print(m_no[1].page_content)
print("\nMedium with overlap : ")
print(m_with[0].page_content)
print("-"*50)
print(m_with[1].page_content)

Medium no overlap : 
Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669
--------------------------------------------------
. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge. The third period (nearly as long as the other two combined) saw Newton as a highly paid government official in London with little further interest in mathematical research

Medium with overlap : 
Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669
--------------------------------------------------
. The first is his boyhood days from 

In [ ]:
l_no = results["Large / No overlap"]["chunks"]
l_with = results["Large / With overlap"]["chunks"]

# print("=== Content Comparison ===")
print("Large  no overlap : ")
print(l_no[0].page_content)
print("-"*50)
print(l_no[1].page_content)
print("\nLarge  with overlap : ")
print(l_with[0].page_content)
print("-"*50)
print(l_with[1].page_content)

Large  no overlap : 
Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge
--------------------------------------------------
. The third period (nearly as long as the other two combined) saw Newton as a highly paid government official in London with little further interest in mathematical research. Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire

Large  with overlap : 
Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669.

In [ ]:
m_no

[Document(metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content="Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669"),
 Document(metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian

In [ ]:
m_with

[Document(metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content="Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669"),
 Document(metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second 

## 3.3 Redesigned test queries

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Redesigned test queries
# อธิบาย: แต่ละ query ออกแบบโดย
#           1. อ่าน document จริงก่อน
#           2. หา paragraph ที่มีข้อมูลครบ
#           3. เลือก keyword จาก paragraph นั้น
#           4. ตั้ง query ให้ semantic ตรงกับ paragraph นั้น
#
# source_hint: บอกว่า keyword อยู่หน้าไหนใน document
#              เพื่อให้ผู้เรียนตรวจสอบได้
# ─────────────────────────────────────────────────────────────────

test_queries = [
    {
        "query"      : "When and where was Newton born?",
        "keywords"   : ["1643", "Woolsthorpe", "Lincolnshire"],
        "source_hint": "Page 1 — paragraph 2: 'Isaac Newton was born in the manor house...'",
    },
    {
        "query"      : "What did Newton study at Cambridge University?",
        "keywords"   : ["Cambridge", "Aristotle", "Descartes", "law degree"],
        "source_hint": "Page 2 — paragraph 4: 'Newton's aim at Cambridge was a law degree...'",
    },
    {
        "query"      : "What did Newton discover during the plague years?",
        "keywords"   : ["plague", "calculus", "optics", "astronomy"],
        "source_hint": "Page 3 — paragraph 1: 'plague closed the University...revolutionary advances'",
    },
    {
        "query"      : "What book did Newton publish in 1687 and what was it about?",
        "keywords"   : ["Principia", "1687", "Halley", "motion"],
        "source_hint": "Page 5 — paragraph 1: 'Halley persuaded Newton...Principia'",
    },
    {
        "query"      : "What positions did Newton hold at the Royal Mint?",
        "keywords"   : ["Mint", "Warden", "Master", "1696"],
        "source_hint": "Page 6 — paragraph 1: 'Warden of the Royal Mint in 1696 and Master in 1699'",
    },
]


## 3.4 Similarity, Retrieval & Scoring Functions

In [ ]:
# Define the 'Similarity' metric (Cosine Similarity)

def cosine_similarity(vec1: list, vec2: list) -> float:
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    # Cosine Similarity Formula: (A · B) / (||A|| * ||B||)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

In [ ]:
# This function acts as the 'Search Engine' for data.

def retrieve_top_k(query: str, chunks: list, chunk_embs: list, k: int = 3) -> list:
    # Convert the input query into a numerical vector (Embedding)
    query_vector = embeddings.embed_query(query)

    # Initialize a list to store similarity scores for each chunk
    all_scored_chunks = []

    # Iterate through each knowledge chunk to compare it with the query
    for i in range(len(chunks)):
        vector = chunk_embs[i]
        content = chunks[i]

        # Calculate the similarity score (1.0 = highly relevant, 0.0 = unrelated)
        score = cosine_similarity(query_vector, vector)

        # Store the result as a tuple (score, content)
        all_scored_chunks.append((score, content))

    # Sort the results in descending order (highest score first)
    all_scored_chunks.sort(key=lambda x: x[0], reverse=True)

    # Extract the 'Top-K' results based on the highest relevance scores
    top_k_results = all_scored_chunks[:k]

    return top_k_results

In [ ]:
# Calculates the accuracy of the retrieval by checking how many
# required keywords exist within the retrieved text chunks.

def keyword_hit_rate(retrieved_results: list, target_keywords: list) -> float:

    # Consolidate all retrieved Top-K content into a single lowercase string
    all_text = ""
    for score, chunk in retrieved_results:
        all_text = all_text + " " + chunk.page_content.lower()

    # Check for the presence of each target keyword in the consolidated text
    found_count = 0
    for word in target_keywords:
        if word.lower() in all_text:
            found_count = found_count + 1

    # Calculate the Hit Rate percentage
    score_percentage = found_count / len(target_keywords)

    return score_percentage

## 3.5 Run Evaluation

In [ ]:
# --- STEP 5: Running the Battle of Strategies ---
# This loop sends each test query to all chunking strategies and records the scores.

# Create a dictionary to store scores for final summary
score_summary = {name: [] for name in strategies}

print("\n=== Retrieval Quality Evaluation: The Ultimate Test ===")

for q_data in test_queries:
    query = q_data["query"]
    keywords = q_data["keywords"]
    # source_hint = q_data["source_hint"]

    # Visual separators for each query
    print(f"\n{'─'*80}")
    print(f"Testing Query: '{query}'")
    print(f"Goal Keywords: {keywords}")
    print(f"{'─'*80}")
    print(f"{'Strategy':<25} {'Score':>6}  {'Sim':>5}  Top result (Snippet)")
    print("-" * 80)

    # Test every strategy against the same query
    for name, data in results.items():
        # 1. Search for top-3 relevant chunks
        retrieved = retrieve_top_k(query, data["chunks"], data["embeddings"], k=3)

        # 2. Check how many keywords were found in those chunks
        score = keyword_hit_rate(retrieved, keywords)

        # 3. Get the confidence score and text of the best chunk
        top_sim = retrieved[0][0]
        top_text = retrieved[0][1].page_content[:80].replace("\n", " ")

        # Save score for the final ranking step
        score_summary[name].append(score)

        # Visual feedback: Use icons for quick interpretation
        rating = "✅" if score >= 0.75 else "⚠️" if score >= 0.50 else "❌"

        # Print the row for this strategy
        print(f"{name:<25} {score:>5.0%} {rating}  {top_sim:.3f}  '{top_text}...'")


=== Retrieval Quality Evaluation: The Ultimate Test ===

────────────────────────────────────────────────────────────────────────────────
Testing Query: 'When and where was Newton born?'
Goal Keywords: ['1643', 'Woolsthorpe', 'Lincolnshire']
────────────────────────────────────────────────────────────────────────────────
Strategy                   Score    Sim  Top result (Snippet)
--------------------------------------------------------------------------------
Small / No overlap         100% ✅  0.700  '. Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Linc...'
Small / With overlap        67% ⚠️  0.700  '. Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Linc...'
Medium / No overlap         67% ⚠️  0.705  '. (The Gregorian calendar was not adopted in England until 1752.) Isaac Newton c...'
Medium / With overlap       67% ⚠️  0.705  '. (The Gregorian calendar was not adopted in England until 1752.) Isaac Newton c...'
Large / No overlap